<a href="https://colab.research.google.com/github/KashyapKrishnan09/SolarModel/blob/main/SolarPanel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Maintenance Alert Logic: Rule-Based Approach

We will define a rule-based system to trigger maintenance alerts. The primary indicators for potential issues will be:

1.  **Significant drop in Actual DC Power compared to Ideal DC Power**: If the `DC_POWER` is considerably lower than the `IDEAL_DC_POWER` (e.g., less than 80% of ideal), it could indicate an issue.
2.  **Low Inverter Efficiency**: If the `INVERTER_EFFICIENCY` falls below a certain threshold (e.g., 0.90 or 90%), it might signal an inverter malfunction or degradation.

We will create a function that applies these rules to each data point and flags potential maintenance alerts.

In [ ]:
def generate_maintenance_alerts(df, ideal_dc_power_threshold=0.8, inverter_efficiency_threshold=0.90):
    """
    Generates maintenance alerts based on DC Power deviation from ideal and inverter efficiency.

    Args:
        df (pd.DataFrame): The DataFrame containing 'DC_POWER', 'IDEAL_DC_POWER', and 'INVERTER_EFFICIENCY'.
        ideal_dc_power_threshold (float): The threshold for DC_POWER / IDEAL_DC_POWER ratio to trigger an alert.
                                        An alert is triggered if DC_POWER < ideal_dc_power_threshold * IDEAL_DC_POWER.
        inverter_efficiency_threshold (float): The threshold for INVERTER_EFFICIENCY to trigger an alert.
                                             An alert is triggered if INVERTER_EFFICIENCY < inverter_efficiency_threshold.

    Returns:
        pd.DataFrame: The DataFrame with an 'ALERT' column indicating maintenance alerts.
    """
    df_alerts = df.copy()
    df_alerts['ALERT'] = False

    # Rule 1: DC Power significantly lower than Ideal DC Power
    # Only apply this rule when IDEAL_DC_POWER is positive (i.e., when there is irradiation)
    condition_dc_power_drop = (df_alerts['IDEAL_DC_POWER'] > 0) & \
                              (df_alerts['DC_POWER'] < ideal_dc_power_threshold * df_alerts['IDEAL_DC_POWER'])
    df_alerts.loc[condition_dc_power_drop, 'ALERT'] = True

    # Rule 2: Low Inverter Efficiency
    # Only apply this rule when DC_POWER is positive (i.e., when inverters are active)
    condition_low_efficiency = (df_alerts['DC_POWER'] > 0) & \
                               (df_alerts['INVERTER_EFFICIENCY'] < inverter_efficiency_threshold)
    df_alerts.loc[condition_low_efficiency, 'ALERT'] = True

    return df_alerts

# Apply the alert logic to Plant 1 data
merged_plant1_data_alerts = generate_maintenance_alerts(merged_plant1_data)

# Apply the alert logic to Plant 2 data
merged_plant2_data_alerts = generate_maintenance_alerts(merged_plant2_data)

### Maintenance Alert Logic: Machine Learning (ML) Based Approach

To more robustly detect performance drops, especially considering factors like ambient temperature, we will use previously trained linear regression models. These models predict DC_POWER based on environmental factors.

Our ML-based alert system will operate as follows:

1.  **Predict Expected DC Power**: For each data point, we will use the appropriate linear regression model (`model_plant1` or `model_plant2`) to predict the `DC_POWER` based on its environmental factors.
2.  **Compare Actual vs. Predicted**: An alert will be triggered if the actual `DC_POWER` falls below a certain percentage of this `predicted_DC_POWER`.

This approach allows for a more nuanced and context-aware alert system than a simple rule-based threshold.

In [ ]:
print('--- Executing all prerequisite setup steps ---')

import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# --- Code from cell d94941ec (Data Cleaning and Loading) ---
print('\n--- Starting Data Cleaning and Loading ---')
plant1_generation_path = '/content/Plant_1_Generation_Data.csv'
plant2_generation_path = '/content/Plant_2_Generation_Data.csv'
plant1_weather_path = '/content/Plant_1_Weather_Sensor_Data.csv'
plant2_weather_path = '/content/Plant_2_Weather_Sensor_Data.csv'

print('Loading dataframes for cleaning...')

try:
    df_gen1 = pd.read_csv(plant1_generation_path)
    df_gen1['DATE_TIME'] = pd.to_datetime(df_gen1['DATE_TIME'], errors='coerce', dayfirst=True)
    df_gen1.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError as e:
    print(f'Error loading Plant 1 Generation data: {e}. Initializing as empty DataFrame.')
    df_gen1 = pd.DataFrame()

try:
    df_weather1 = pd.read_csv(plant1_weather_path)
    df_weather1['DATE_TIME'] = pd.to_datetime(df_weather1['DATE_TIME'], errors='coerce')
    df_weather1.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError as e:
    print(f'Error loading Plant 1 Weather data: {e}. Initializing as empty DataFrame.')
    df_weather1 = pd.DataFrame()

try:
    df_gen2 = pd.read_csv(plant2_generation_path)
    df_gen2['DATE_TIME'] = pd.to_datetime(df_gen2['DATE_TIME'], errors='coerce')
    df_gen2.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError as e:
    print(f'Error loading Plant 2 Generation data: {e}. Initializing as empty DataFrame.')
    df_gen2 = pd.DataFrame()

try:
    df_weather2 = pd.read_csv(plant2_weather_path)
    df_weather2['DATE_TIME'] = pd.to_datetime(df_weather2['DATE_TIME'], errors='coerce')
    df_weather2.dropna(subset=['DATE_TIME'], inplace=True)
except FileNotFoundError as e:
    print(f'Error loading Plant 2 Weather data: {e}. Initializing as empty DataFrame.')
    df_weather2 = pd.DataFrame()

print('Dataframe loading complete.')

def clean_dataframe(df, file_name):
    numeric_cols = df.select_dtypes(include=np.number).columns

    if not numeric_cols.empty:
        df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both')
        df[numeric_cols] = df[numeric_cols].ffill().bfill()

    return df

if not df_gen1.empty:
    df_gen1 = clean_dataframe(df_gen1.copy(), 'Plant_1_Generation_Data.csv')
if not df_weather1.empty:
    df_weather1 = clean_dataframe(df_weather1.copy(), 'Plant_1_Weather_Sensor_Data.csv')
if not df_gen2.empty:
    df_gen2 = clean_dataframe(df_gen2.copy(), 'Plant_2_Generation_Data.csv')
if not df_weather2.empty:
    df_weather2 = clean_dataframe(df_weather2.copy(), 'Plant_2_Weather_Sensor_Data.csv')

print('\nData cleaning complete. Dataframes are now cleaned.')

# --- Code from cell f0b2da74 (Merge data and calculate efficiency) ---
print('\n--- Starting Data Merging and Efficiency Calculation ---')
merged_plant1_data = pd.merge(
    df_gen1[['DATE_TIME', 'PLANT_ID', 'DC_POWER', 'AC_POWER', 'TOTAL_YIELD']],
    df_weather1[['DATE_TIME', 'PLANT_ID', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']],
    on=['DATE_TIME', 'PLANT_ID'],
    how='inner'
)
merged_plant1_data['INVERTER_EFFICIENCY'] = np.where(
    merged_plant1_data['DC_POWER'] > 0,
    merged_plant1_data['AC_POWER'] / merged_plant1_data['DC_POWER'],
    0
).clip(max=1.0)

merged_plant2_data = pd.merge(
    df_gen2[['DATE_TIME', 'PLANT_ID', 'DC_POWER', 'AC_POWER', 'TOTAL_YIELD']],
    df_weather2[['DATE_TIME', 'PLANT_ID', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']],
    on=['DATE_TIME', 'PLANT_ID'],
    how='inner'
)
merged_plant2_data['INVERTER_EFFICIENCY'] = np.where(
    merged_plant2_data['DC_POWER'] > 0,
    merged_plant2_data['AC_POWER'] / merged_plant2_data['DC_POWER'],
    0
).clip(max=1.0)
print('Data merging and efficiency calculation complete.')

# --- Code from cell 16c127ea (Calculate Ideal DC Power) ---
print('\n--- Starting Ideal DC Power Calculation ---')
plant1_power_per_irr = merged_plant1_data[
    (merged_plant1_data['IRRADIATION'] > 0) & (merged_plant1_data['DC_POWER'] > 0)
].copy()

if not plant1_power_per_irr.empty:
    plant1_power_per_irr['DC_POWER_PER_IRRADIATION'] = plant1_power_per_irr['DC_POWER'] / plant1_power_per_irr['IRRADIATION']
    max_dc_per_irr_plant1 = plant1_power_per_irr['DC_POWER_PER_IRRADIATION'].max()
    merged_plant1_data['IDEAL_DC_POWER'] = merged_plant1_data['IRRADIATION'] * max_dc_per_irr_plant1
else:
    merged_plant1_data['IDEAL_DC_POWER'] = 0

plant2_power_per_irr = merged_plant2_data[
    (merged_plant2_data['IRRADIATION'] > 0) & (merged_plant2_data['DC_POWER'] > 0)
].copy()

if not plant2_power_per_irr.empty:
    plant2_power_per_irr['DC_POWER_PER_IRRADIATION'] = plant2_power_per_irr['DC_POWER'] / plant2_power_per_irr['IRRADIATION']
    max_dc_per_irr_plant2 = plant2_power_per_irr['DC_POWER_PER_IRRADIATION'].max()
    merged_plant2_data['IDEAL_DC_POWER'] = merged_plant2_data['IRRADIATION'] * max_dc_per_irr_plant2
else:
    merged_plant2_data['IDEAL_DC_POWER'] = 0
print('Ideal DC Power calculation complete.')

# --- Code from cell 0f75c019 (Simulate Noisy Weather Data) ---
print('\n--- Starting Noisy Weather Data Simulation ---')
def add_weather_noise(df, irradiation_noise_std=0.05, temp_noise_std=2.0, seed=42):
    np.random.seed(seed)
    noisy_df = df.copy()

    if 'IRRADIATION' in noisy_df.columns:
        noise = np.random.normal(0, irradiation_noise_std, size=len(noisy_df))
        noisy_df['IRRADIATION'] = noisy_df['IRRADIATION'] + noise
        noisy_df['IRRADIATION'] = noisy_df['IRRADIATION'].clip(lower=0)

    if 'AMBIENT_TEMPERATURE' in noisy_df.columns:
        noise = np.random.normal(0, temp_noise_std, size=len(noisy_df))
        noisy_df['AMBIENT_TEMPERATURE'] = noisy_df['AMBIENT_TEMPERATURE'] + noise

    if 'MODULE_TEMPERATURE' in noisy_df.columns:
        noise = np.random.normal(0, temp_noise_std, size=len(noisy_df))
        noisy_df['MODULE_TEMPERATURE'] = noisy_df['MODULE_TEMPERATURE'] + noise

    return noisy_df

noisy_plant1_data = add_weather_noise(merged_plant1_data.copy(), irradiation_noise_std=0.01, temp_noise_std=1.0)
noisy_plant1_data['INVERTER_EFFICIENCY'] = np.where(
    noisy_plant1_data['DC_POWER'] > 0,
    noisy_plant1_data['AC_POWER'] / noisy_plant1_data['DC_POWER'],
    0
).clip(max=1.0)
noisy_plant1_data['IDEAL_DC_POWER'] = noisy_plant1_data['IRRADIATION'] * max_dc_per_irr_plant1

noisy_plant2_data = add_weather_noise(merged_plant2_data.copy(), irradiation_noise_std=0.005, temp_noise_std=0.5)
noisy_plant2_data['INVERTER_EFFICIENCY'] = np.where(
    noisy_plant2_data['DC_POWER'] > 0,
    noisy_plant2_data['AC_POWER'] / noisy_plant2_data['DC_POWER'],
    0
).clip(max=1.0)
noisy_plant2_data['IDEAL_DC_POWER'] = noisy_plant2_data['IRRADIATION'] * max_dc_per_irr_plant2
print('Noisy weather data simulation complete.')

# --- Code from cell 2996764d (Prepare Data for Linear Regression Models) ---
print('\n--- Starting Data Preparation for Linear Regression Models ---')
X_plant1 = merged_plant1_data[['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']]
y_plant1 = merged_plant1_data['DC_POWER']
X_plant1 = X_plant1.dropna()
y_plant1 = y_plant1[X_plant1.index]
X_train_plant1, X_test_plant1, y_train_plant1, y_test_plant1 = train_test_split(X_plant1, y_plant1, test_size=0.2, random_state=42)

X_plant2 = merged_plant2_data[['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']]
y_plant2 = merged_plant2_data['DC_POWER']
X_plant2 = X_plant2.dropna()
y_plant2 = y_plant2[X_plant2.index]
X_train_plant2, X_test_plant2, y_train_plant2, y_test_plant2 = train_test_split(X_plant2, y_plant2, test_size=0.2, random_state=42)
print('Data preparation for linear regression models complete.')

# --- Code from cell 65de977e (Train and Evaluate Linear Regression Models) ---
print('\n--- Starting Linear Regression Model Training and Evaluation ---')
model_plant1 = LinearRegression()
model_plant1.fit(X_train_plant1, y_train_plant1)

y_pred_plant1 = model_plant1.predict(X_test_plant1)
r2_plant1 = r2_score(y_test_plant1, y_pred_plant1)
print(f'Plant 1 - R-squared: {r2_plant1:.4f}')

model_plant2 = LinearRegression()
model_plant2.fit(X_train_plant2, y_train_plant2)

y_pred_plant2 = model_plant2.predict(X_test_plant2)
r2_plant2 = r2_score(y_test_plant2, y_pred_plant2)
print(f'Plant 2 - R-squared: {r2_plant2:.4f}')
print('Linear regression model training and evaluation complete.')

print('\n--- All prerequisite setup steps executed successfully! ---')

In [ ]:
def generate_ml_maintenance_alerts(df, model, features, prediction_threshold_ratio=0.85):
    """
    Generates maintenance alerts based on a machine learning model's prediction of DC Power.

    Args:
        df (pd.DataFrame): The DataFrame containing 'DC_POWER' and the 'features' used by the model.
        model (sklearn.linear_model.LinearRegression): The trained linear regression model.
        features (list): A list of column names in df that correspond to the model's features.
        prediction_threshold_ratio (float): The ratio of actual DC_POWER to predicted DC_POWER.
                                        An alert is triggered if DC_POWER < prediction_threshold_ratio * predicted_DC_POWER.

    Returns:
        pd.DataFrame: The DataFrame with an 'ML_ALERT' column indicating maintenance alerts.
    """
    df_ml_alerts = df.copy()
    df_ml_alerts['ML_ALERT'] = False

    # Only make predictions where IRRADIATION is positive, as DC_POWER is 0 when no light
    active_generation_indices = df_ml_alerts[df_ml_alerts['IRRADIATION'] > 0].index

    if not active_generation_indices.empty:
        # Prepare features for prediction
        X_predict = df_ml_alerts.loc[active_generation_indices, features]

        # Predict expected DC_POWER
        df_ml_alerts.loc[active_generation_indices, 'PREDICTED_DC_POWER'] = model.predict(X_predict)

        # Rule: Actual DC_POWER significantly lower than predicted DC_POWER
        condition_ml_alert = ((df_ml_alerts['DC_POWER'] < prediction_threshold_ratio * df_ml_alerts['PREDICTED_DC_POWER']) &
                             (df_ml_alerts['DC_POWER'] > 0))
        df_ml_alerts.loc[condition_ml_alert, 'ML_ALERT'] = True
    else:
        df_ml_alerts['PREDICTED_DC_POWER'] = 0

    return df_ml_alerts

# Define the features used by the models
model_features = ['IRRADIATION', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE']

# Apply ML alert logic to Plant 1 data (original)
ml_alerts_original_plant1 = generate_ml_maintenance_alerts(merged_plant1_data.copy(), model_plant1, model_features)

# Apply ML alert logic to Plant 2 data (original)
ml_alerts_original_plant2 = generate_ml_maintenance_alerts(merged_plant2_data.copy(), model_plant2, model_features)

# Apply ML alert logic to Plant 1 data (noisy)
ml_alerts_noisy_plant1 = generate_ml_maintenance_alerts(noisy_plant1_data.copy(), model_plant1, model_features)

# Apply ML alert logic to Plant 2 data (noisy)
ml_alerts_noisy_plant2 = generate_ml_maintenance_alerts(noisy_plant2_data.copy(), model_plant2, model_features)